# SAM3 Diagnostic Notebook

This notebook exposes **image inference**, **video inference**, and **visualization** as independent cells.

**Purpose**: Determine whether empty masks are caused by the bfloat16 / CPU-offload modifications or by the original code.

**How to use**:
1. Run Cell 1 (Setup) and Cell 2 (Config)
2. Toggle `USE_BFLOAT16` / `OFFLOAD_VIDEO_CPU` in Cell 2
3. Run the inference + visualization cells you need
4. Compare results between flag combinations

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 1 — Setup & Install
# ═══════════════════════════════════════════════════════════════════════

import os, sys

# Detect Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    # Clone and install SAM3 if not already present
    if not os.path.exists('sam3'):
        !git clone https://github.com/facebookresearch/sam3.git
    %cd sam3
    !pip install -e . -q
    !pip install opencv-python matplotlib -q
else:
    # Local: make sure we're in the repo root
    repo_root = os.path.dirname(os.path.abspath('__file__'))
    if repo_root not in sys.path:
        sys.path.insert(0, repo_root)

import torch
print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU            : {torch.cuda.get_device_name(0)}')
    print(f'VRAM           : {torch.cuda.get_device_properties(0).total_mem / 1024**3:.2f} GiB')
    !nvidia-smi

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 2 — Configuration Flags
# ═══════════════════════════════════════════════════════════════════════
#
# Toggle these to test different modes:
#   - Both False  → original unmodified SAM3 behavior
#   - Both True   → VRAM-optimized path (bfloat16 + CPU offload)
#   - Mix         → isolate which flag causes issues
#
# ───────────────────────────────────────────────────────────────────────

USE_BFLOAT16       = False   # Cast model weights to bfloat16 (halves VRAM)
OFFLOAD_VIDEO_CPU  = False   # Keep video frames on CPU until needed
DEVICE             = "cuda"  # "cuda" or "cpu"
CONFIDENCE_THRESH  = 0.5     # Detection confidence threshold

# ─── Inputs ───
IMAGE_PATH         = "inputs/images/sample_image.jpg"
VIDEO_PATH         = "inputs/videos/3-dogs.mp4"
TEXT_PROMPT_IMAGE   = "shoe"
TEXT_PROMPT_VIDEO   = "dogs"

print("=" * 60)
print("CONFIGURATION")
print("=" * 60)
print(f"  USE_BFLOAT16      = {USE_BFLOAT16}")
print(f"  OFFLOAD_VIDEO_CPU = {OFFLOAD_VIDEO_CPU}")
print(f"  DEVICE            = {DEVICE}")
print(f"  CONFIDENCE_THRESH = {CONFIDENCE_THRESH}")
print(f"  IMAGE_PATH        = {IMAGE_PATH}")
print(f"  VIDEO_PATH        = {VIDEO_PATH}")
print(f"  TEXT_PROMPT_IMAGE  = {TEXT_PROMPT_IMAGE}")
print(f"  TEXT_PROMPT_VIDEO  = {TEXT_PROMPT_VIDEO}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 3 — Image Model: Build & Inference
# ═══════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from PIL import Image
from sam3.model_builder import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor

print("[Image] Building model...")
image_model = build_sam3_image_model()

if USE_BFLOAT16:
    print("[Image] Casting model to bfloat16...")
    image_model.to(dtype=torch.bfloat16)

image_model.to(device=DEVICE)
print(f"[Image] Model dtype: {next(image_model.parameters()).dtype}")
print(f"[Image] Model device: {next(image_model.parameters()).device}")

processor = Sam3Processor(image_model, confidence_threshold=CONFIDENCE_THRESH, device=DEVICE)

print(f"\n[Image] Loading image: {IMAGE_PATH}")
pil_image = Image.open(IMAGE_PATH)
print(f"[Image] Image size: {pil_image.size}")

inference_state = processor.set_image(pil_image)

print(f"[Image] Running inference with prompt: '{TEXT_PROMPT_IMAGE}'...")
inference_state = processor.set_text_prompt(state=inference_state, prompt=TEXT_PROMPT_IMAGE)

img_masks  = inference_state["masks"]
img_boxes  = inference_state["boxes"]
img_scores = inference_state["scores"]

print("\n" + "=" * 60)
print("IMAGE INFERENCE RESULTS")
print("=" * 60)
print(f"  Masks found    : {len(img_masks)}")
if len(img_masks) > 0:
    print(f"  Masks shape    : {img_masks.shape}")
    print(f"  Masks dtype    : {img_masks.dtype}")
    for i in range(min(len(img_masks), 5)):
        m = img_masks[i]
        if hasattr(m, 'cpu'):
            m = m.cpu()
        nz = m.sum().item()
        print(f"  Mask {i}: score={img_scores[i].item():.4f}  non-zero pixels={nz}  shape={m.shape}")
    if all((img_masks[i].cpu().sum().item() == 0) for i in range(len(img_masks))):
        print("\n  ⚠  WARNING: ALL masks are entirely zero!")
    else:
        print("\n  ✓  At least one mask has non-zero pixels.")
else:
    print("  ⚠  WARNING: No masks detected at all!")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 4 — Video Model: Build & Inference
# ═══════════════════════════════════════════════════════════════════════

import torch
import numpy as np
from sam3.model_builder import build_sam3_video_predictor

print("[Video] Building predictor...")
video_predictor = build_sam3_video_predictor()

if USE_BFLOAT16:
    print("[Video] Casting model to bfloat16...")
    video_predictor.model.to(dtype=torch.bfloat16)

video_predictor.model.to(device=DEVICE)
print(f"[Video] Model dtype: {next(video_predictor.model.parameters()).dtype}")
print(f"[Video] Model device: {next(video_predictor.model.parameters()).device}")

print(f"\n[Video] Starting session for: {VIDEO_PATH}")
print(f"[Video] offload_video_to_cpu = {OFFLOAD_VIDEO_CPU}")
response = video_predictor.handle_request(
    request=dict(
        type="start_session",
        resource_path=VIDEO_PATH,
        offload_video_to_cpu=OFFLOAD_VIDEO_CPU,
    )
)
session_id = response["session_id"]
print(f"[Video] Session started: {session_id}")

print(f"[Video] Adding prompt: '{TEXT_PROMPT_VIDEO}' on frame 0...")
response = video_predictor.handle_request(
    request=dict(
        type="add_prompt",
        session_id=session_id,
        frame_index=0,
        text=TEXT_PROMPT_VIDEO,
    )
)
vid_output = response["outputs"]

print("\n" + "=" * 60)
print("VIDEO INFERENCE RESULTS")
print("=" * 60)
if isinstance(vid_output, dict):
    for key, val in vid_output.items():
        if hasattr(val, 'shape'):
            print(f"  '{key}' → shape={val.shape}  dtype={val.dtype}")
        elif isinstance(val, dict):
            print(f"  '{key}' → {val}")
        else:
            print(f"  '{key}' → {type(val).__name__}: {val}")
    
    # Check masks
    masks_key = None
    for k in ['out_binary_masks', 'masks']:
        if k in vid_output:
            masks_key = k
            break
    if masks_key is not None:
        masks = vid_output[masks_key]
        if hasattr(masks, 'cpu'):
            masks = masks.cpu().numpy()
        if masks.size == 0:
            print("\n  ⚠  WARNING: Mask array is EMPTY (size=0)!")
        elif masks.sum() == 0:
            print("\n  ⚠  WARNING: Masks exist but are ALL ZEROS!")
        else:
            print(f"\n  ✓  Masks have {masks.sum()} non-zero pixels")
    
    # Check obj_ids
    if 'out_obj_ids' in vid_output:
        ids = vid_output['out_obj_ids']
        if hasattr(ids, '__len__') and len(ids) == 0:
            print("  ⚠  WARNING: out_obj_ids is EMPTY — model tracked 0 objects!")
        else:
            print(f"  ✓  Tracked object IDs: {ids}")
else:
    print(f"  Output type: {type(vid_output)}")
    print(f"  Output: {vid_output}")
print("=" * 60)

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 5 — Visualizer: Image Mask Overlay
# ═══════════════════════════════════════════════════════════════════════
# Requires: Cell 3 (img_masks, IMAGE_PATH)

import cv2
import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

orig_img = np.array(Image.open(IMAGE_PATH))
fig, axes = plt.subplots(1, min(len(img_masks) + 1, 4), figsize=(20, 6))
if not isinstance(axes, np.ndarray):
    axes = [axes]

# Original image
axes[0].imshow(orig_img)
axes[0].set_title('Original')
axes[0].axis('off')

# Overlay each mask (up to 3)
for i in range(min(len(img_masks), 3)):
    mask = img_masks[i]
    if hasattr(mask, 'cpu'):
        mask = mask.cpu().numpy()
    mask = np.squeeze(mask)
    
    # Resize mask to match image if needed
    h, w = orig_img.shape[:2]
    if mask.shape != (h, w):
        mask = cv2.resize(mask.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)
    
    overlay = orig_img.copy()
    colored_mask = np.zeros_like(overlay)
    colored_mask[:, :, 0] = 255  # Red in RGB
    binary = mask > 0
    overlay[binary] = (overlay[binary] * 0.5 + colored_mask[binary] * 0.5).astype(np.uint8)
    
    score = img_scores[i].item() if hasattr(img_scores[i], 'item') else img_scores[i]
    axes[i + 1].imshow(overlay)
    axes[i + 1].set_title(f'Mask {i} (score={score:.3f})')
    axes[i + 1].axis('off')

plt.suptitle(f'Image: {IMAGE_PATH} | Prompt: "{TEXT_PROMPT_IMAGE}" | bfloat16={USE_BFLOAT16}', fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 6 — Visualizer: Video Mask Overlay
# ═══════════════════════════════════════════════════════════════════════
# Requires: Cell 4 (vid_output, VIDEO_PATH)

import cv2
import numpy as np
import os
from IPython.display import HTML, display
import base64

def extract_mask_from_output(output, frame_idx=0):
    """Extract a 2D binary mask for a given frame from the video output dict."""
    mask = None
    if isinstance(output, dict):
        if 'out_binary_masks' in output:
            m = output['out_binary_masks']
            if hasattr(m, 'cpu'):
                m = m.cpu().numpy()
            if m.size > 0:
                m = np.squeeze(m)
                if m.ndim > 2:
                    mask = m[0]  # Take first object
                else:
                    mask = m
    elif isinstance(output, list) and frame_idx < len(output):
        frame_data = output[frame_idx]
        if isinstance(frame_data, dict):
            for key in ['out_binary_masks', 'masks']:
                if key in frame_data:
                    m = frame_data[key]
                    if hasattr(m, 'cpu'):
                        m = m.cpu().numpy()
                    if m.size > 0:
                        m = np.squeeze(m)
                        mask = m[0] if m.ndim > 2 else m
                    break
    return mask

# Read video and overlay masks
cap = cv2.VideoCapture(VIDEO_PATH)
fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

os.makedirs('outputs/overlay', exist_ok=True)
out_path = 'outputs/overlay/video_overlay.mp4'
fourcc = cv2.VideoWriter_fourcc(*'mp4v')
writer = cv2.VideoWriter(out_path, fourcc, fps, (width, height))

frame_idx = 0
frames_with_mask = 0
while True:
    ret, frame = cap.read()
    if not ret:
        break
    
    mask = extract_mask_from_output(vid_output, frame_idx)
    if mask is not None:
        if mask.shape != (height, width):
            mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST)
        colored = np.zeros_like(frame)
        colored[:, :, 2] = 255  # Red in BGR
        binary = mask > 0
        if binary.any():
            frames_with_mask += 1
            frame[binary] = cv2.addWeighted(frame, 0.5, colored, 0.5, 0)[binary]
    
    writer.write(frame)
    frame_idx += 1

cap.release()
writer.release()

print(f"Processed {frame_idx} frames, {frames_with_mask} had non-empty masks.")
print(f"Saved overlay video to: {out_path}")

# Display in notebook (Colab)
if os.path.exists(out_path):
    with open(out_path, 'rb') as f:
        video_data = f.read()
    b64 = base64.b64encode(video_data).decode()
    display(HTML(f'''
    <video width="640" controls>
        <source src="data:video/mp4;base64,{b64}" type="video/mp4">
    </video>
    <p>Overlay: {VIDEO_PATH} | Prompt: "{TEXT_PROMPT_VIDEO}" | bfloat16={USE_BFLOAT16} | offload={OFFLOAD_VIDEO_CPU}</p>
    '''))

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 7 — Pure Mask Export (Image & Video)
# ═══════════════════════════════════════════════════════════════════════
# Saves raw masks for offline inspection.
# Requires: Cell 3 (img_masks) and/or Cell 4 (vid_output)

import os, torch
import numpy as np
from PIL import Image as PILImage
import cv2

os.makedirs('outputs/diagnostic', exist_ok=True)

# ─── Image mask export ───
try:
    if len(img_masks) > 0:
        mask = img_masks[0]
        if hasattr(mask, 'cpu'):
            mask = mask.cpu().numpy()
        mask = np.squeeze(mask)
        mask_img = PILImage.fromarray((mask * 255).astype(np.uint8))
        save_path = 'outputs/diagnostic/image_mask.png'
        mask_img.save(save_path)
        print(f"✓ Saved image mask to {save_path}  (non-zero: {mask.sum()})")
    else:
        print("⚠ No image masks to export (run Cell 3 first)")
except NameError:
    print("⚠ img_masks not defined — run Cell 3 first")

# ─── Video mask export (.pt) ───
try:
    save_path = 'outputs/diagnostic/video_output.pt'
    torch.save(vid_output, save_path)
    print(f"✓ Saved video output to {save_path}")
except NameError:
    print("⚠ vid_output not defined — run Cell 4 first")

# ─── Video mask-only video (white on black) ───
try:
    cap = cv2.VideoCapture(VIDEO_PATH)
    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    
    mask_vid_path = 'outputs/diagnostic/video_mask_only.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    writer = cv2.VideoWriter(mask_vid_path, fourcc, fps, (width, height))
    
    frame_idx = 0
    non_empty = 0
    while True:
        ret, _ = cap.read()
        if not ret:
            break
        
        mask = extract_mask_from_output(vid_output, frame_idx)
        mask_frame = np.zeros((height, width, 3), dtype=np.uint8)
        if mask is not None:
            if mask.shape != (height, width):
                mask = cv2.resize(mask.astype(np.uint8), (width, height), interpolation=cv2.INTER_NEAREST)
            binary = mask > 0
            if binary.any():
                non_empty += 1
            mask_frame[binary] = [255, 255, 255]
        
        writer.write(mask_frame)
        frame_idx += 1
    
    cap.release()
    writer.release()
    print(f"✓ Saved mask-only video to {mask_vid_path}  ({non_empty}/{frame_idx} frames with mask)")
except NameError:
    print("⚠ vid_output not defined — run Cell 4 and Cell 6 first")

In [ ]:
# ═══════════════════════════════════════════════════════════════════════
# Cell 8 — Cleanup (optional)
# ═══════════════════════════════════════════════════════════════════════
# Free GPU memory between test runs

import gc, torch

# Delete models from memory
for var in ['image_model', 'processor', 'video_predictor']:
    if var in dir():
        exec(f'del {var}')

gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print(f"GPU memory after cleanup: {torch.cuda.memory_allocated() / 1024**3:.2f} GiB allocated")
    print(f"                          {torch.cuda.memory_reserved() / 1024**3:.2f} GiB reserved")
print("Done. Change flags in Cell 2 and re-run from Cell 3/4.")